# Flight Data Platform - Camada Gold

Consome as tabelas silver e constrói agregações e KPIs prontos para análise e visualização: aeronaves por país, distribuição de heading, estatísticas de altitude/velocidade e partidas por aeroporto/linha aérea.

In [0]:
from pyspark.sql.functions import *
import builtins
import tempfile
import json
import os
import requests
import datetime
import time
from google.oauth2 import service_account
import google.auth.transport.requests as auth_requests

## 1. Leitura das tabelas silver

In [0]:
# Leitura das tabelas silver
silver_states = spark.table("workspace.default.silver_opensky_states")
silver_airports = spark.table("workspace.default.silver_dim_airports")
silver_departures = spark.table("workspace.default.silver_opensky_departures")

print(f"silver_states: {silver_states.count()} linhas")
print(f"silver_airports: {silver_airports.count()} linhas")
print(f"silver_departures: {silver_departures.count()} linhas")

## 2. Agregações e KPIs

In [0]:
# --- KPI 1: Aeronaves por país ---
gold_aircraft_by_country = (
    silver_states
    .groupBy("origin_country")
    .agg(
        count("*").alias("total_aircraft"),
        sum(when(col("is_airborne") == True, 1).otherwise(0)).alias("airborne_count"),
        sum(when(col("is_airborne") == False, 1).otherwise(0)).alias("on_ground_count"),
        round(avg("altitude_ft"), 2).alias("avg_altitude_ft"),
        round(avg("velocity_kmh"), 2).alias("avg_velocity_kmh"),
    )
    .orderBy(col("total_aircraft").desc())
)

print(f"gold_aircraft_by_country: {gold_aircraft_by_country.count()} países")
display(gold_aircraft_by_country.limit(20))

In [0]:
# --- KPI 2: Distribuição de heading ---
gold_heading_distribution = (
    silver_states
    .filter(col("heading_category").isNotNull())
    .groupBy("heading_category")
    .agg(
        count("*").alias("aircraft_count"),
        round(avg("velocity_kmh"), 2).alias("avg_velocity_kmh"),
        round(avg("altitude_ft"), 2).alias("avg_altitude_ft"),
    )
    .orderBy("heading_category")
)

print(f"gold_heading_distribution: {gold_heading_distribution.count()} categorias")
display(gold_heading_distribution)

In [0]:
# --- KPI 3: Partidas por aeroporto e linha aérea ---
# Extrai o prefixo da linha aérea do callsign (primeiros 3 caracteres)
gold_departures_summary = (
    silver_departures
    .withColumn("airline_code", upper(substring(col("callsign"), 1, 3)))
    .groupBy("departure_icao", "departure_airport_name", "departure_city", "airline_code")
    .agg(
        count("*").alias("departure_count"),
    )
    .orderBy(col("departure_count").desc())
)

print(f"gold_departures_summary: {gold_departures_summary.count()} grupos")
display(gold_departures_summary.limit(20))

In [0]:
# --- KPI 4: Estatísticas de aeroportos ---
# Combina silver_airports com contagem de aeronaves no ar por país
gold_airport_stats = (
    silver_airports
    .filter(col("icao_code").isNotNull())
    .groupBy("iso_country", "type")
    .agg(
        count("*").alias("airport_count"),
        round(avg("elevation_ft"), 2).alias("avg_elevation_ft"),
        max("elevation_ft").alias("max_elevation_ft"),
        min("elevation_ft").alias("min_elevation_ft"),
    )
    .orderBy(col("airport_count").desc())
)

print(f"gold_airport_stats: {gold_airport_stats.count()} grupos")
display(gold_airport_stats.limit(20))

## 3. Exportação para o Google BigQuery (Reverse ETL)

Exporta as tabelas gold do Databricks (AWS) para o BigQuery (GCP) usando a **BigQuery REST API** via `google-auth` + `requests`.

### Sandbox (projeto `engdta`)

O projeto `engdta` é um sandbox GCP **sem billing ativo** (`billingEnabled=false`). Por isso:

* A escrita usa **load jobs com NDJSON inline** via BigQuery REST API (sem Spark connector, sem bucket GCS). O sandbox bloqueia streaming insert e DML, mas load jobs são permitidos.
* As tabelas criadas no dataset `flight_data` **expiram em 60 dias** — comportamento esperado do sandbox, não tente remover a expiração.
* O parâmetro `temporaryGcsBucket` **não é utilizado** na escrita.

> **Nota técnica**: O Spark BigQuery Connector (`spark-bigquery-with-dependencies_2.12:0.36.1`) não é suportado para escrita em compute serverless (`UNSUPPORTED_DATA_SOURCE_WRITE`). A abordagem REST API funciona tanto em serverless quanto em cluster clássico.

### Pré-requisitos

1. **GCP Service Account**: crie uma service account no GCP com a role `BigQuery Data Editor`
2. **Databricks Secret Scope**: armazene o JSON da service account em `gcp-secrets` / key: `service-account-key`
3. **BigQuery Dataset**: o dataset `flight_data` (location US) deve existir no projeto `engdta`
4. **Bibliotecas Python**: `google-auth` e `requests` (já disponíveis no Databricks Runtime)

### Migração para projeto com billing

Ao migrar para um projeto GCP com billing ativo e cluster clássico, basta:
1. Instalar o Spark BigQuery Connector Maven (`com.google.cloud.spark:spark-bigquery-with-dependencies_2.12:0.36.1`)
2. Criar um bucket GCS temporário e adicionar `GCS_TEMP_BUCKET` no `.env`
3. Substituir a célula de exportação para usar `.format("bigquery").option("temporaryGcsBucket", GCS_TEMP_BUCKET)` (modo indireto)
4. O modo indireto é mais eficiente para grandes volumes pois usa o GCS como staging

In [0]:
# --- Configuração do BigQuery: leitura e validação do .env ---
_env_path = "/Workspace/Users/jcesarsantana215@gmail.com/Flight-Data-Platform/.env"

# Parser do .env — tolerante a linhas vazias e comentários (#)
config = {}
with open(_env_path) as f:
    for line in f:
        line = line.strip()
        if not line or line.startswith("#"):
            continue
        if "=" in line:
            k, v = line.split("=", 1)
            config[k.strip()] = v.strip()

# Validar chaves obrigatórias
REQUIRED_KEYS = ["GCP_PROJECT_ID", "BQ_DATASET", "GCS_TEMP_BUCKET", "SECRET_SCOPE", "SECRET_KEY"]
missing = [k for k in REQUIRED_KEYS if k not in config]
if missing:
    raise ValueError(f"Chaves ausentes no .env: {missing}. Verifique: {_env_path}")

GCP_PROJECT_ID = config["GCP_PROJECT_ID"]
BQ_DATASET = config["BQ_DATASET"]
GCS_TEMP_BUCKET = config["GCS_TEMP_BUCKET"]  # lido mas não usado no sandbox
SECRET_SCOPE = config["SECRET_SCOPE"]
SECRET_KEY = config["SECRET_KEY"]

# Validar valores esperados (não aceitar placeholders)
if GCP_PROJECT_ID != "engdta":
    raise ValueError(
        f"GCP_PROJECT_ID deve ser 'engdta' (sandbox), encontrado: '{GCP_PROJECT_ID}'. "
        f"Atualize o arquivo .env."
    )
if BQ_DATASET != "flight_data":
    raise ValueError(
        f"BQ_DATASET deve ser 'flight_data', encontrado: '{BQ_DATASET}'. "
        f"Atualize o arquivo .env."
    )

print(f".env validado: project={GCP_PROJECT_ID}, dataset={BQ_DATASET}")

In [0]:
# --- Service account key e credenciais BigQuery ---
# Ler a service account key do Databricks Secrets
try:
    key_raw = dbutils.secrets.get(scope=SECRET_SCOPE, key=SECRET_KEY)
except Exception as e:
    raise ValueError(
        f"Não foi possível ler o secret '{SECRET_KEY}' do scope '{SECRET_SCOPE}'. "
        f"Crie o scope com: databricks secrets create-scope {SECRET_SCOPE} "
        f"e adicione a key com: databricks secrets put-secret {SECRET_SCOPE} {SECRET_KEY}. "
        f"Erro original: {e}"
    )

# Extrair JSON válido (o secret pode conter output de terminal antes do JSON)
brace_idx = key_raw.find('{')
if brace_idx < 0:
    raise ValueError("O secret não contém um JSON válido de service account.")
key_json = key_raw[brace_idx:]

# Escrever a key em arquivo temporário (compatibilidade com Spark connector em cluster clássico)
key_path = tempfile.NamedTemporaryFile(mode="w", suffix=".json", delete=False)
key_path.write(key_json)
key_path.close()

# Criar credenciais para BigQuery REST API (funciona em serverless sem o connector)
bq_credentials = service_account.Credentials.from_service_account_info(
    json.loads(key_json),
    scopes=["https://www.googleapis.com/auth/bigquery"],
)
_auth_req = auth_requests.Request()
bq_credentials.refresh(_auth_req)

print(f"Credenciais BigQuery criadas e token renovado ✓")

In [0]:
# --- Resumo da configuração BigQuery ---
# Tentar configurar Spark conf (funciona em cluster clássico, silencioso em serverless)
for conf_key, conf_val in [("parentProject", GCP_PROJECT_ID), ("credentialsFile", key_path.name)]:
    try:
        spark.conf.set(conf_key, conf_val)
    except Exception:
        pass  # Spark Connect: credenciais usadas via REST API

print(f"Configuração BigQuery (sandbox):")
print(f"  project     = {GCP_PROJECT_ID}")
print(f"  dataset     = {BQ_DATASET}")
print(f"  writeMethod = load job NDJSON inline (sem GCS, sem billing)")
print(f"  GCS bucket  = {GCS_TEMP_BUCKET} (não usado no sandbox)")
print(f"Service account key carregada de Databricks Secrets ✓")

In [0]:
# --- BigQuery: mapeamento de tipos e helper de schema ---

# Mapeamento de tipos Spark → BigQuery
_BQ_TYPE_MAP = {
    "string": "STRING",
    "long": "INT64",
    "int": "INT64",
    "integer": "INT64",
    "double": "FLOAT64",
    "float": "FLOAT64",
    "boolean": "BOOL",
    "timestamp": "TIMESTAMP",
    "date": "DATE",
}

def _spark_schema_to_bq(df):
    """Converte o schema do Spark DataFrame para o formato do BigQuery."""
    fields = []
    for field in df.schema.fields:
        spark_type = field.dataType.simpleString()
        bq_type = _BQ_TYPE_MAP.get(spark_type, "STRING")
        fields.append({"name": field.name, "type": bq_type})
    return {"fields": fields}

In [0]:
# --- Função: escrever Spark DataFrame em tabela BigQuery via load job ---

def _write_df_to_bq(df, table_name, project_id, dataset_id, creds):
    """Escreve um Spark DataFrame em uma tabela BigQuery via load job.
    Usa upload multipart com NDJSON inline — funciona no sandbox sem billing
    (streaming insert e DML são bloqueados no free tier)."""
    base_url = "https://bigquery.googleapis.com/bigquery/v2"

    # Garantir token válido
    if creds.expired:
        _req = auth_requests.Request()
        creds.refresh(_req)

    headers = {"Authorization": f"Bearer {creds.token}"}

    # Deletar tabela existente (modo overwrite)
    requests.delete(
        f"{base_url}/projects/{project_id}/datasets/{dataset_id}/tables/{table_name}",
        headers=headers,
        timeout=30,
    )

    # Coletar linhas e schema
    rows = [row.asDict() for row in df.collect()]
    schema = _spark_schema_to_bq(df)

    if not rows:
        # Tabela vazia: criar apenas com schema
        create_body = {
            "tableReference": {
                "projectId": project_id,
                "datasetId": dataset_id,
                "tableId": table_name,
            },
            "schema": schema,
        }
        resp = requests.post(
            f"{base_url}/projects/{project_id}/datasets/{dataset_id}/tables",
            headers={**headers, "Content-Type": "application/json"},
            json=create_body,
            timeout=30,
        )
        if resp.status_code not in (200, 201):
            raise RuntimeError(f"Erro ao criar tabela vazia {project_id}.{dataset_id}.{table_name}: {resp.text}")
        return

    # Serializar linhas para NDJSON (converter datetime/date para string)
    def _serialize_val(val):
        if isinstance(val, datetime.datetime):
            return val.strftime("%Y-%m-%dT%H:%M:%S.%fZ")
        if isinstance(val, datetime.date):
            return val.isoformat()
        return val

    ndjson_lines = []
    for row in rows:
        serialized = {k: _serialize_val(v) for k, v in row.items()}
        ndjson_lines.append(json.dumps(serialized))
    ndjson_data = "\n".join(ndjson_lines)

    # Criar load job via upload multipart (NDJSON inline)
    job_config = {
        "configuration": {
            "load": {
                "destinationTable": {
                    "projectId": project_id,
                    "datasetId": dataset_id,
                    "tableId": table_name,
                },
                "sourceFormat": "NEWLINE_DELIMITED_JSON",
                "writeDisposition": "WRITE_TRUNCATE",
                "schema": schema,
                "autodetect": False,
            }
        }
    }

    boundary = "bq-boundary-" + os.urandom(8).hex()
    body = (
        f"--{boundary}\r\n"
        f"Content-Type: application/json\r\n\r\n"
        f"{json.dumps(job_config)}\r\n"
        f"--{boundary}\r\n"
        f"Content-Type: application/octet-stream\r\n\r\n"
        f"{ndjson_data}\r\n"
        f"--{boundary}--"
    )

    upload_url = f"https://bigquery.googleapis.com/upload/bigquery/v2/projects/{project_id}/jobs?uploadType=multipart"
    resp = requests.post(
        upload_url,
        headers={
            **headers,
            "Content-Type": f"multipart/related; boundary={boundary}",
        },
        data=body.encode("utf-8"),
        timeout=120,
    )
    if resp.status_code not in (200, 201):
        raise RuntimeError(f"Load job falhou para {project_id}.{dataset_id}.{table_name}: {resp.text}")

    # Aguardar conclusão do job (poll)
    job_id = resp.json().get("jobReference", {}).get("jobId")
    if not job_id:
        raise RuntimeError(f"Load job sem jobId para {project_id}.{dataset_id}.{table_name}: {resp.text}")

    for _ in range(60):  # max 120 segundos
        time.sleep(2)
        poll = requests.get(
            f"{base_url}/projects/{project_id}/jobs/{job_id}",
            headers=headers,
            timeout=30,
        )
        if poll.status_code != 200:
            continue
        job_status = poll.json().get("status", {})
        state = job_status.get("state", "UNKNOWN")
        if state == "DONE":
            if job_status.get("errorResult"):
                raise RuntimeError(f"Load job erro em {project_id}.{dataset_id}.{table_name}: {job_status['errorResult']}")
            return  # sucesso!
    raise RuntimeError(f"Load job timeout para {project_id}.{dataset_id}.{table_name} (job {job_id})")

In [0]:
# --- Exportar as 4 tabelas gold para o BigQuery ---

bq_tables = {
    "gold_aircraft_by_country": gold_aircraft_by_country,
    "gold_heading_distribution": gold_heading_distribution,
    "gold_departures_summary": gold_departures_summary,
    "gold_airport_stats": gold_airport_stats,
}

try:
    for table_name, df in bq_tables.items():
        row_count = df.count()
        full_table = f"{GCP_PROJECT_ID}.{BQ_DATASET}.{table_name}"
        print(f"Exportando {table_name} ({row_count} linhas) → {full_table} ...")
        _write_df_to_bq(df, table_name, GCP_PROJECT_ID, BQ_DATASET, bq_credentials)
        print(f"  ✓ {full_table} exportada com sucesso!")

    print("\nTodas as tabelas gold foram exportadas para o BigQuery!")
finally:
    os.unlink(key_path.name)
    print(f"Arquivo temporário da key removido: {key_path.name}")